# Teste Interativo da Classe FlightAnomalyDetector

Notebook de validação da classe `FlightAnomalyDetector` (detecção de aeroportos com perfil atípico).

Pipeline testado:
1. Importação e instanciação
2. Carregamento e agregação por aeroporto
3. Pré-processamento (padronização)
4. Três métodos de detecção: Isolation Forest, LOF, Silhouette KMeans
5. Consenso entre métodos
6. PCA para visualização
7. Gráficos comparativos e perfil dos anômalos

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto para permitir importar a classe do módulo notebooks
sys.path.append(str(Path(".").resolve().parent))

from notebooks.anomalias import FlightAnomalyDetector

In [ ]:
# Inicializa o detector.
# - contamination=0.05 → assume 5% de anomalias (ajustável)
# - k_final=4 → use o mesmo k escolhido no notebook 04 para o silhouette individual
# - consensus_threshold=2 → anomalia confirmada se sinalizada por >=2 dos 3 métodos
detector = FlightAnomalyDetector(
    input_path="../data/processed/flights_model.parquet",
    min_voos=500,
    contamination=0.05,
    n_neighbors=20,
    k_final=4,
    silhouette_threshold=0.0,
    consensus_threshold=2,
)

In [ ]:
# 1. Carregar os dados
df = detector.load_data()

In [ ]:
# 2. Agregar features por aeroporto (com features extras: p95, taxa_atraso_critico)
airport_features = detector.aggregate_by_airport()
airport_features.head(3)

In [ ]:
# 3. Pré-processamento (nulos + padronização)
X_scaled = detector.preprocess_features()

## Métodos de detecção

In [ ]:
# Método 1 — Isolation Forest
detector.run_isolation_forest()

In [ ]:
# Método 2 — Local Outlier Factor
detector.run_lof()

In [ ]:
# Método 3 — Silhouette individual do KMeans (k=4 mesmo do notebook 04)
detector.run_silhouette_anomaly()

## Consenso entre métodos

Aeroportos sinalizados por **pelo menos 2 dos 3 métodos** são considerados anomalias confirmadas.

In [ ]:
# Cruza os três métodos
anomalos = detector.build_consensus()
anomalos[["n_metodos_anomalia", "cluster", "taxa_atraso", "taxa_atraso_grave",
          "media_arrival_delay", "std_arrival_delay", "n_destinos", "qtd_voos"]].round(3)

## Visualizações

In [ ]:
# PCA 2D para projetar os 15 features em 2 dimensões
detector.fit_pca_2d()

In [ ]:
# Comparação visual dos três métodos lado a lado
detector.plot_methods_comparison()

In [ ]:
# Mapa único de consenso (cor = quantos métodos sinalizaram)
detector.plot_consensus_map()

In [ ]:
# Scatter dos scores IF vs LOF com cortes e consenso destacado
detector.plot_iso_vs_lof()

In [ ]:
# Heatmap comparando perfil médio: Anômalos vs Normais
perfil = detector.plot_profile_comparison()

## Alternativa: pipeline completo com `run_all`

In [ ]:
# Roda tudo de uma vez
# detector_full = FlightAnomalyDetector(
#     input_path="../data/processed/flights_model.parquet",
# )
# detector_full.run_all(show_plots=True)

## Experimentar parâmetros diferentes

Alguns experimentos que valem investigar:

In [ ]:
# Mais agressivo: contamination=0.10 marca 10% como anômalos
# detector_agressivo = FlightAnomalyDetector(
#     input_path="../data/processed/flights_model.parquet",
#     contamination=0.10,
# )
# detector_agressivo.run_all(show_plots=False)
# detector_agressivo.anomalos_consenso

In [ ]:
# Mais conservador: exigir consenso entre os 3 métodos
# detector_conservador = FlightAnomalyDetector(
#     input_path="../data/processed/flights_model.parquet",
#     consensus_threshold=3,
# )
# detector_conservador.run_all(show_plots=False)
# detector_conservador.anomalos_consenso